# Semantic retrieval benchmark — multilingual-e5-base

Objective comparison of **lexical** vs **semantic** retrieval quality for Mobile_mem0's memory model, using `groonga/multilingual-e5-base-Q4_K_M-GGUF` (Q4_K_M, 768 dimensions, mean pooling, `"query: "`/`"passage: "` prefixes) as the reference embedding model — already verified working on-device; see `download_model.py`'s own doc comment for why this repo, not cstr's original one.

Standalone: no Android, no JNI, no Firestore, and no changes to Mobile_mem0's production code. See `benchmark/README.md` in the repo for the full picture — this notebook just runs the same steps `run_benchmark.py` does, cell by cell, entirely on Colab's free CPU tier.

Steps 1–9 are Part A: the lexical-vs-semantic (base) comparison. Steps 10–13 extend that into Part B/C: a hybrid-ranking semantic-weight sweep, a per-weight robustness/bootstrap check, and a final `RECOMMENDED_SEMANTIC_WEIGHT` — the number that actually goes into production's `AppContainer.SEMANTIC_RANKING_WEIGHT`.

Run every cell top to bottom. Nothing here needs a Hugging Face token (the model repo is public), and nothing downloaded here is written back into the repo.

## 1. Clone the repo and install dependencies

In [ ]:
import os

if os.path.isdir("Mobile_mem0"):
    # Already cloned in this runtime from an earlier cell run — pull the
    # latest instead of re-cloning, so a fix pushed after this session
    # started (e.g. a corrected model repo) actually takes effect. A plain
    # re-run of this cell alone is NOT enough after a git pull, though:
    # Python caches already-imported modules, so also do Runtime -> Restart
    # session, then Run all again, any time the benchmark's own .py files
    # changed upstream.
    !cd Mobile_mem0 && git pull
else:
    !git clone --depth 1 https://github.com/rmant7/Mobile_mem0.git

%cd Mobile_mem0/benchmark
# --extra-index-url pulls a prebuilt CPU wheel from llama-cpp-python's own
# release index instead of compiling llama.cpp from source via CMake, which
# otherwise takes 15-25+ minutes on Colab's shared CPU for no benefit here
# (this benchmark never needs GPU offload). --prefer-binary makes pip take
# that wheel over a newer sdist if one exists; falls back to compiling only
# if this exact Python/platform combo genuinely has no prebuilt wheel.
%pip install -q --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu llama-cpp-python
%pip install -q huggingface_hub

## 2. Download the GGUF from Hugging Face
Resolves the actual filename at run time (never hardcoded) — see `download_model.py`. Lands in `huggingface_hub`'s own cache on Colab's ephemeral disk, never committed anywhere.

In [ ]:
from download_model import download, resolve_filename

print("Resolved file:", resolve_filename())
model_path = download()
print("Downloaded to:", model_path)

## 3. Load the dataset

In [ ]:
from run_benchmark import load_jsonl, DEFAULT_DATASET_DIR

memories = load_jsonl(DEFAULT_DATASET_DIR / "memories.jsonl")
queries = load_jsonl(DEFAULT_DATASET_DIR / "queries.jsonl")
print(f"{len(memories)} memories, {len(queries)} queries")

from collections import Counter
print("Queries by category:", Counter(q["category"] for q in queries))

## 4. Load the model and build embeddings
Every memory gets `"passage: "`, every query gets `"query: "` — see `embedder.py`. This is the slow cell (CPU inference on Colab's shared cores); expect a few minutes for ~105 memories + ~100 queries.

In [ ]:
from embedder import E5Embedder

embedder = E5Embedder(model_path)
print("Embedding dimension:", embedder.dimension)
assert embedder.dimension == 768, f"expected 768-dimensional embeddings, got {embedder.dimension}"

memory_vectors = {m["id"]: embedder.embed_passage(m["text"]) for m in memories}
query_vectors = {q["id"]: embedder.embed_query(q["text"]) for q in queries}
print(f"Embedded {len(memory_vectors)} memories and {len(query_vectors)} queries")

## 5. Cosine similarity + semantic retrieval
Ranks every memory against every query by cosine similarity — a direct sort for measurement purposes only, not a candidate ranking formula for production.

In [ ]:
from metrics import QueryResult, cosine_similarity

semantic_results = []
for q in queries:
    qvec = query_vectors[q["id"]]
    scored = [(mid, cosine_similarity(qvec, vec)) for mid, vec in memory_vectors.items()]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    semantic_results.append(QueryResult(
        query_id=q["id"], category=q["category"],
        ranked_ids=[mid for mid, _ in scored],
        ground_truth_ids=q["ground_truth_ids"],
        scores_by_id=dict(scored),
    ))
print(f"Ranked {len(semantic_results)} queries semantically")

## 6. Lexical retrieval
Python port of Mobile_mem0's own `MemoryRanking.tokenize()`/`overlap()` — see `lexical.py`.

In [ ]:
import lexical

lexical_results = []
for q in queries:
    hits = lexical.rank(q["text"], memories)
    lexical_results.append(QueryResult(
        query_id=q["id"], category=q["category"],
        ranked_ids=[h.memory_id for h in hits],
        ground_truth_ids=q["ground_truth_ids"],
    ))
print(f"Ranked {len(lexical_results)} queries lexically")

## 7. Overall comparison table

In [ ]:
from metrics import aggregate, aggregate_by_category
from run_benchmark import print_table

lex_overall = aggregate(lexical_results)
sem_overall = aggregate(semantic_results)
print_table("Overall", {"lexical": lex_overall, "semantic (base)": sem_overall})
print(f"\nSemantic (base) avg cosine — positive: {sem_overall.avg_cosine_positive:.3f}, negative: {sem_overall.avg_cosine_negative:.3f}")

## 8. Breakdown by category
`low_overlap` is the category that most directly answers whether semantic retrieval adds anything lexical search can't already do; `identifier` is where lexical is expected to hold its own or win.

In [ ]:
lex_by_cat = aggregate_by_category(lexical_results)
sem_by_cat = aggregate_by_category(semantic_results)
for cat in sorted(lex_by_cat.keys()):
    print_table(f"Category: {cat}", {"lexical": lex_by_cat[cat], "semantic (base)": sem_by_cat[cat]})

## 9. Save results (JSON + CSV)

In [ ]:
from pathlib import Path
from run_benchmark import save_results

save_results(
    Path("results"),
    overall={"lexical": lex_overall.as_dict(), "semantic_base": sem_overall.as_dict()},
    by_category={
        "lexical": {c: m.as_dict() for c, m in lex_by_cat.items()},
        "semantic_base": {c: m.as_dict() for c, m in sem_by_cat.items()},
    },
)

# In Colab: download the two files to your machine, or read them back with
# `from google.colab import files; files.download("results/results.csv")`.

## 10. Hybrid ranking sweep (Part B/C)

Extends Part A with the semantic-weight sweep needed to actually tune
production's `SEMANTIC_RANKING_WEIGHT`. Reuses `memory_vectors`/`query_vectors`
from step 4 and `memories`/`queries` from step 3 — no re-embedding, no
re-download. `hybrid.py`'s `hybrid_rank()` merges lexical hits (recomputed
here, cheaply, with real scores this time — step 6 only kept ranked ids) and
the top semantic candidates into one weighted-sum ranking, mirroring the
*shape* of `ai.localstudio.commercialmemory.HeuristicContextRanker` with this
benchmark's own two signals. Every other production weight stays fixed; only
the semantic weight varies, across `0.00, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50`.

In [ ]:
import lexical
from metrics import cosine_similarity

raw = []
for q in queries:
    lexical_hits = lexical.rank(q["text"], memories)
    qvec = query_vectors[q["id"]]
    scored = [(mid, cosine_similarity(qvec, vec)) for mid, vec in memory_vectors.items()]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    raw.append({"query": q, "lexical_hits": lexical_hits, "semantic_scored": scored})
print(f"Prepared raw lexical+semantic candidates for {len(raw)} queries")

In [ ]:
from bootstrap import bootstrap_mrr
from run_hybrid_sweep import SEMANTIC_WEIGHTS, WATCH_CATEGORIES, run_hybrid_for_weight, print_sweep_table, print_category_tables

sweep = {}
for w in SEMANTIC_WEIGHTS:
    hybrid_results = run_hybrid_for_weight(raw, w)
    overall = aggregate(hybrid_results)
    by_cat = aggregate_by_category(hybrid_results)
    boot = bootstrap_mrr(hybrid_results)
    sweep[w] = {
        "overall": overall.as_dict(),
        "by_category": {c: m.as_dict() for c, m in by_cat.items()},
        "bootstrap": boot,
    }
    print(f"weight={w:.2f}: MRR={overall.mrr:.3f} (bootstrap std={boot['mrr_std']:.4f}), Recall@1={overall.recall_at_1:.3f}")

print_sweep_table(sweep)

## 11. Category breakdown per weight

The categories this task asked to watch explicitly for every swept weight —
not just Recall@1, but whether `identifier`/`exact` (near-perfect for
lexical already) ever regress, and how `low_overlap` (semantic's own best
category) moves.

In [ ]:
print_category_tables(sweep, WATCH_CATEGORIES)

## 12. Robustness check + recommendation

`recommend.py`'s `recommend_weight()` is a fixed, pure decision rule (unit
tested in `tests/test_hybrid.py` on synthetic data): a candidate weight must
beat lexical-only MRR/Recall@1, must not hurt `identifier`/`exact`
recall_at_1 by more than a small tolerance, and a move to a larger weight
must clear its own bootstrap noise band — otherwise the result is reported
as ambiguous and this cell falls back to the existing `0.20` baseline rather
than guessing between `0.30`/`0.35`, per this task's own explicit rule.

In [ ]:
from recommend import recommend_weight

recommendation = recommend_weight(
    lexical_overall=lex_overall.as_dict(),
    lexical_by_category={c: m.as_dict() for c, m in lex_by_cat.items()},
    sweep=sweep,
)
print(recommendation["reason"])
print(f"RECOMMENDED_SEMANTIC_WEIGHT = {recommendation['weight']}")

## 13. Save hybrid sweep results (JSON)

In [ ]:
import json
from pathlib import Path

out_dir = Path("results")
out_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "lexical_overall": lex_overall.as_dict(),
    "lexical_by_category": {c: m.as_dict() for c, m in lex_by_cat.items()},
    "sweep": {str(w): v for w, v in sweep.items()},
    "recommendation": recommendation,
}
out_path = out_dir / "hybrid_sweep_results.json"
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {out_path}")

# In Colab: from google.colab import files; files.download(str(out_path))